In [2]:
import zipfile
import os

zip_path = "/lakehouse/default/Files/archive (1).zip"
extract_path = "/lakehouse/default/Files/bronze/olist"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP file extracted successfully!")
print(os.listdir(extract_path))

StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 6, Finished, Available, Finished, False)

ZIP file extracted successfully!
['product_category_name_translation.csv', 'olist_order_reviews_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_customers_dataset.csv', 'olist_order_items_dataset.csv', 'olist_orders_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_products_dataset.csv']


In [3]:
import pandas as pd
import os

path = "/lakehouse/default/Files/bronze/olist"

files = os.listdir(path)

for file in files:
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(path, file))

        print("\n" + "=" * 60)
        print("FILE:", file)
        print("ROWS AND COLUMNS:", df.shape)
        print("COLUMNS:", list(df.columns))
        print("MISSING VALUES:", df.isnull().sum().sum())

        if "order_purchase_timestamp" in df.columns:
            print("DATE RANGE:")
            print(df["order_purchase_timestamp"].min())
            print(df["order_purchase_timestamp"].max())

print("\nALL FILES CHECKED!")

StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 7, Finished, Available, Finished, False)


FILE: product_category_name_translation.csv
ROWS AND COLUMNS: (71, 2)
COLUMNS: ['product_category_name', 'product_category_name_english']
MISSING VALUES: 0

FILE: olist_order_reviews_dataset.csv
ROWS AND COLUMNS: (99224, 7)
COLUMNS: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
MISSING VALUES: 145903

FILE: olist_geolocation_dataset.csv
ROWS AND COLUMNS: (1000163, 5)
COLUMNS: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
MISSING VALUES: 0

FILE: olist_customers_dataset.csv
ROWS AND COLUMNS: (99441, 5)
COLUMNS: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
MISSING VALUES: 0

FILE: olist_order_items_dataset.csv
ROWS AND COLUMNS: (112650, 7)
COLUMNS: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
MISSING VALUES: 0

F

In [4]:
import pandas as pd
import os

path = "/lakehouse/default/Files/bronze/olist"

orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))

# Միավորում ենք աղյուսակները
df = orders.merge(customers, on="customer_id")
df = df.merge(items, on="order_id")

# Թողնում ենք միայն ավարտված պատվերները
df = df[df["order_status"] == "delivered"].copy()

# Ամսաթվի ձևափոխում
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

# Հաշվում ենք գործարքի գումարը
df["total_amount"] = df["price"]

# Պահպանում ենք արդյունքը
output_path = "/lakehouse/default/Files/silver"

os.makedirs(output_path, exist_ok=True)

df.to_parquet(
    os.path.join(output_path, "olist_valid_transactions.parquet"),
    index=False
)

print("Final dataset shape:", df.shape)
print("Unique customers:", df["customer_unique_id"].nunique())
print("Date range:",
      df["order_purchase_timestamp"].min(),
      "to",
      df["order_purchase_timestamp"].max())

print("SAVED SUCCESSFULLY!")

StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 8, Finished, Available, Finished, False)

Final dataset shape: (110197, 19)
Unique customers: 93358
Date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37
SAVED SUCCESSFULLY!


In [5]:
import pandas as pd
import os

path = "/lakehouse/default/Files/silver/olist_valid_transactions.parquet"

df = pd.read_parquet(path)

# Հաշվարկի վերջնական ամսաթիվ
snapshot_date = df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

# RFM ցուցանիշներ
rfm = df.groupby("customer_unique_id").agg(
    Recency=("order_purchase_timestamp",
             lambda x: (snapshot_date - x.max()).days),

    Frequency=("order_id", "nunique"),

    Monetary=("price", "sum")
).reset_index()

# Պահպանում
rfm.to_parquet(
    "/lakehouse/default/Files/silver/customer_rfm.parquet",
    index=False
)

print("RFM shape:", rfm.shape)
print(rfm.describe())
print("\nRFM SAVED SUCCESSFULLY!")


StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 9, Finished, Available, Finished, False)

RFM shape: (93358, 4)
            Recency     Frequency      Monetary
count  93358.000000  93358.000000  93358.000000
mean     237.941773      1.033420    141.621480
std      152.591453      0.209097    215.694014
min        1.000000      1.000000      0.850000
25%      114.000000      1.000000     47.650000
50%      219.000000      1.000000     89.730000
75%      346.000000      1.000000    154.737500
max      714.000000     15.000000  13440.000000

RFM SAVED SUCCESSFULLY!


In [6]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
import joblib

warnings.filterwarnings("ignore")

# 1. PATHS AND DATA LOADING


base_path = "/lakehouse/default/Files"
data_path = base_path + "/bronze/olist"
silver_path = base_path + "/silver"

os.makedirs(silver_path, exist_ok=True)

orders = pd.read_csv(
    data_path + "/olist_orders_dataset.csv"
)

customers = pd.read_csv(
    data_path + "/olist_customers_dataset.csv"
)

items = pd.read_csv(
    data_path + "/olist_order_items_dataset.csv"
)

# ------------------------------------------------------------
# 2. DATA CLEANING AND MERGING
# ------------------------------------------------------------

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

# Keep completed orders only
orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_purchase_timestamp"].notna())
].copy()

# Merge orders, customers and order items
df = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="inner"
)

df = df.merge(
    items[["order_id", "product_id", "price"]],
    on="order_id",
    how="inner"
)

# Keep valid transaction rows
df = df[
    (df["price"] > 0) &
    (df["customer_unique_id"].notna())
].copy()

df["total_amount"] = df["price"]

df.to_parquet(
    silver_path + "/olist_valid_transactions.parquet",
    index=False
)

print("1. DATA CLEANING COMPLETED")
print("Transactions:", len(df))
print("Unique customers:", df["customer_unique_id"].nunique())

# ------------------------------------------------------------
# 3. CREATE CUSTOMER FEATURES AND FUTURE PURCHASE LABELS
# ------------------------------------------------------------

# Each order item row represents one listed item.
# Olist does not provide an explicit quantity column.

min_date = df["order_purchase_timestamp"].min()
max_date = df["order_purchase_timestamp"].max()

snapshot_dates = pd.date_range(
    start=min_date + pd.Timedelta(days=90),
    end=max_date - pd.Timedelta(days=30),
    freq="30D"
)

all_snapshots = []

for snapshot_date in snapshot_dates:

    # Previous 90 days
    history = df[
        (df["order_purchase_timestamp"] < snapshot_date) &
        (df["order_purchase_timestamp"] >=
         snapshot_date - pd.Timedelta(days=90))
    ].copy()

    # Next 30 days
    future = df[
        (df["order_purchase_timestamp"] >= snapshot_date) &
        (df["order_purchase_timestamp"] <
         snapshot_date + pd.Timedelta(days=30))
    ]

    if history.empty:
        continue

    # Calculate order amounts
    order_amounts = history.groupby(
        ["customer_unique_id", "order_id"]
    )["price"].sum().reset_index(name="order_amount")

    avg_order = order_amounts.groupby(
        "customer_unique_id"
    )["order_amount"].mean().rename("AvgOrderValue")

    # Customer features
    features = history.groupby("customer_unique_id").agg(
        Recency=(
            "order_purchase_timestamp",
            lambda x: (snapshot_date - x.max()).days
        ),
        Frequency=("order_id", "nunique"),
        Monetary=("price", "sum"),
        DistinctProducts=("product_id", "nunique"),
        TotalItems=("order_id", "count")
    )

    features = features.join(avg_order)

    # Label: at least one delivered purchase in next 30 days
    future_customers = set(
        future["customer_unique_id"].unique()
    )

    features["Target"] = (
        features.index.isin(future_customers)
    ).astype(int)

    features["SnapshotDate"] = snapshot_date

    all_snapshots.append(features.reset_index())

dataset = pd.concat(all_snapshots, ignore_index=True)

feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "AvgOrderValue",
    "DistinctProducts",
    "TotalItems"
]

dataset[feature_columns] = dataset[feature_columns].fillna(0)

dataset.to_parquet(
    silver_path + "/customer_classification_dataset.parquet",
    index=False
)

print("\n2. CLASSIFICATION DATASET CREATED")
print("Rows:", len(dataset))
print("Snapshots:", dataset["SnapshotDate"].nunique())
print("Positive labels:", dataset["Target"].sum())
print("Positive rate:", round(dataset["Target"].mean(), 4))

# ------------------------------------------------------------
# 4. TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ------------------------------------------------------------

dates = sorted(dataset["SnapshotDate"].unique())

if len(dates) < 8:
    raise ValueError(
        "Not enough snapshot dates for reliable temporal evaluation."
    )

train_end = int(len(dates) * 0.60)
validation_end = int(len(dates) * 0.80)

# Leave one 30-day snapshot gap between periods
train_dates = dates[:train_end]
validation_dates = dates[train_end + 1:validation_end]
test_dates = dates[validation_end + 1:]

train = dataset[
    dataset["SnapshotDate"].isin(train_dates)
].copy()

validation = dataset[
    dataset["SnapshotDate"].isin(validation_dates)
].copy()

test = dataset[
    dataset["SnapshotDate"].isin(test_dates)
].copy()

if train.empty or validation.empty or test.empty:
    raise ValueError("Train, validation or test dataset is empty.")

X_train = train[feature_columns]
y_train = train["Target"]

X_val = validation[feature_columns]
y_val = validation["Target"]

X_test = test[feature_columns]
y_test = test["Target"]

print("\n3. TEMPORAL DATA SPLIT")
print("Train:", train["SnapshotDate"].min(), "to",
      train["SnapshotDate"].max())
print("Validation:", validation["SnapshotDate"].min(), "to",
      validation["SnapshotDate"].max())
print("Test:", test["SnapshotDate"].min(), "to",
      test["SnapshotDate"].max())

# ------------------------------------------------------------
# 5. CLASSIFICATION MODELS
# ------------------------------------------------------------

models = {
    "DummyClassifier": DummyClassifier(strategy="prior"),

    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

# Add LightGBM if it is already installed
try:
    from lightgbm import LGBMClassifier

    models["LightGBM"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        verbosity=-1
    )

except ImportError:
    print("\nLightGBM is not installed. "
          "Dummy and Logistic Regression will be compared.")

validation_results = []
trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, probabilities)

    validation_results.append({
        "Model": name,
        "Validation_Average_Precision": ap
    })

    trained_models[name] = model

validation_results = pd.DataFrame(validation_results)

print("\n4. VALIDATION RESULTS")
print(validation_results)

# Select best model using Average Precision
best_name = validation_results.sort_values(
    "Validation_Average_Precision",
    ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_name]

# ------------------------------------------------------------
# 6. SELECT CLASSIFICATION THRESHOLD ON VALIDATION DATA
# ------------------------------------------------------------

from sklearn.metrics import precision_recall_curve

val_probabilities = best_model.predict_proba(X_val)[:, 1]

precision_values, recall_values, thresholds = (
    precision_recall_curve(y_val, val_probabilities)
)

if len(thresholds) > 0:
    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1] /
        (precision_values[:-1] + recall_values[:-1] + 1e-10)
    )

    best_threshold = thresholds[np.argmax(f1_values)]
else:
    best_threshold = 0.5

# ------------------------------------------------------------
# 7. FINAL TEST EVALUATION
# ------------------------------------------------------------

test_probabilities = best_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

test_ap = average_precision_score(
    y_test, test_probabilities
)

test_roc = (
    roc_auc_score(y_test, test_probabilities)
    if y_test.nunique() == 2 else np.nan
)

test_precision = precision_score(
    y_test, test_predictions, zero_division=0
)

test_recall = recall_score(
    y_test, test_predictions, zero_division=0
)

test_f1 = f1_score(
    y_test, test_predictions, zero_division=0
)

print("\n5. FINAL CLASSIFICATION RESULTS")
print("Selected model:", best_name)
print("Selected threshold:", round(float(best_threshold), 4))
print("Average Precision:", round(test_ap, 4))
print("ROC-AUC:", round(test_roc, 4))
print("Precision:", round(test_precision, 4))
print("Recall:", round(test_recall, 4))
print("F1-score:", round(test_f1, 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

# Save metrics
test_metrics = pd.DataFrame([{
    "Model": best_name,
    "Threshold": best_threshold,
    "Average_Precision": test_ap,
    "ROC_AUC": test_roc,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1": test_f1
}])

test_metrics.to_csv(
    silver_path + "/classification_metrics.csv",
    index=False
)

# Save trained model
joblib.dump(
    best_model,
    silver_path + "/best_classification_model.pkl"
)

# ------------------------------------------------------------
# 8. RFM CLUSTERING ON THE LATEST 90-DAY SNAPSHOT
# ------------------------------------------------------------

latest_snapshot = pd.Timestamp(snapshot_dates[-1])

rfm = dataset[
    dataset["SnapshotDate"] == latest_snapshot
].copy()

rfm_features = rfm[feature_columns].copy()

# Log transformation and scaling
rfm_transformed = np.log1p(rfm_features)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_transformed)

# Compare KMeans for k = 2 to 8
clustering_results = []

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(rfm_scaled)

    clustering_results.append({
        "K": k,
        "Inertia": kmeans.inertia_,
        "Silhouette": silhouette_score(
            rfm_scaled,
            labels,
            sample_size=min(5000, len(rfm_scaled)),
            random_state=42
        ),
        "Davies_Bouldin": davies_bouldin_score(
            rfm_scaled, labels
        )
    })

clustering_results = pd.DataFrame(clustering_results)

print("\n6. KMEANS COMPARISON")
print(clustering_results)

# Choose the model with highest silhouette score
best_k = int(
    clustering_results.loc[
        clustering_results["Silhouette"].idxmax(), "K"
    ]
)

final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

rfm["Segment"] = final_kmeans.fit_predict(rfm_scaled)

# Segment profiles
segment_profiles = rfm.groupby("Segment").agg(
    Customers=("customer_unique_id", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Median_Recency=("Recency", "median"),
    Median_Frequency=("Frequency", "median"),
    Median_Monetary=("Monetary", "median")
).reset_index()

print("\n7. FINAL CUSTOMER SEGMENTS")
print("Selected number of clusters:", best_k)
print(segment_profiles)

# ------------------------------------------------------------
# 9. SAVE FINAL RESULTS
# ------------------------------------------------------------

rfm.to_parquet(
    silver_path + "/customer_segments.parquet",
    index=False
)

segment_profiles.to_csv(
    silver_path + "/segment_profiles.csv",
    index=False
)

clustering_results.to_csv(
    silver_path + "/kmeans_comparison.csv",
    index=False
)

dataset.to_csv(
    silver_path + "/classification_dataset.csv",
    index=False
)

joblib.dump(
    final_kmeans,
    silver_path + "/final_kmeans_model.pkl"
)

joblib.dump(
    scaler,
    silver_path + "/clustering_scaler.pkl"
)

# Save test predictions
test_output = test[
    ["customer_unique_id", "SnapshotDate", "Target"]
].copy()

test_output["Purchase_Probability"] = test_probabilities
test_output["Predicted_Target"] = test_predictions

test_output.to_csv(
    silver_path + "/test_predictions.csv",
    index=False
)

print("\n" + "=" * 60)
print("FINAL PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)

print("\nSaved files:")
print(os.listdir(silver_path))


StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 10, Finished, Available, Finished, False)

1. DATA CLEANING COMPLETED
Transactions: 110197
Unique customers: 93358

2. CLASSIFICATION DATASET CREATED
Rows: 232248
Snapshots: 20
Positive labels: 919
Positive rate: 0.004

3. TEMPORAL DATA SPLIT
Train: 2016-12-14 12:16:38 to 2017-11-09 12:16:38
Validation: 2018-01-08 12:16:38 to 2018-03-09 12:16:38
Test: 2018-05-08 12:16:38 to 2018-07-07 12:16:38
2026-11-18 21:10:35 [INFO] setting mlflow tracking uri to sds://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow, change this environment variable if you want log to other places
🏃 View run tidy_hair_zvnt8773 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/fb6e8c5c-3769-48a7-9c65-cd05942b08cd
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run sleepy_gyro_jyp8pjy5 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/85a320d4-80fc-4262-9506-58585054d718
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run olive_pump_zd665mr7 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/c4b627cd-2a40-4abd-b7e1-4f70f03ea76e
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c



4. VALIDATION RESULTS
                Model  Validation_Average_Precision
0     DummyClassifier                      0.003509
1  LogisticRegression                      0.023255
2            LightGBM                      0.003939

5. FINAL CLASSIFICATION RESULTS
Selected model: LogisticRegression
Selected threshold: 0.9029
Average Precision: 0.0172
ROC-AUC: 0.6474
Precision: 0.0755
Recall: 0.0183
F1-score: 0.0295
Confusion Matrix:
[[58641    49]
 [  214     4]]
🏃 View run busy_tail_yddqhsfm at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/620cfe11-47ef-495a-96a1-99c062b26e9b
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run careful_drawer_sgdh60zg at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/3d06d817-bbdd-40f7-a873-bc9d6301b6ca
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


... (3 outputs left out)

🏃 View run amusing_pea_6pmt8n65 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/7ce78c3d-8df9-4226-b6d3-6e7cfe574967
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run kind_guava_yq1q4s75 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/e8034b9d-5689-45e4-a14a-8458b140dda5
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c



6. KMEANS COMPARISON
   K       Inertia  Silhouette  Davies_Bouldin
0  2  85185.917939    0.654684        1.117617
1  3  63591.118115    0.326368        1.118284
2  4  48985.966680    0.342099        0.955273
3  5  39435.218066    0.362330        0.859428
4  6  32198.904624    0.392981        0.811172
5  7  26563.705269    0.390759        0.765807
6  8  23972.879477    0.346797        0.817784
🏃 View run blue_salt_wx1scnqd at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/e4857d09-5a1d-46ed-b321-f105d5ce36b2
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c



7. FINAL CUSTOMER SEGMENTS
Selected number of clusters: 2
   Segment  Customers  Avg_Recency  Avg_Frequency  Avg_Monetary  \
0        0       1030    45.723301       1.237864    278.890816   
1        1      17901    46.720407       1.000000    136.500330   

   Median_Recency  Median_Frequency  Median_Monetary  
0            49.0               1.0           173.94  
1            51.0               1.0            89.00  

FINAL PIPELINE COMPLETED SUCCESSFULLY!

Saved files:
['final_kmeans_model.pkl', 'classification_dataset.csv', 'clustering_scaler.pkl', 'classification_metrics.csv', 'test_predictions.csv', 'customer_segments.parquet', 'kmeans_comparison.csv', 'best_classification_model.pkl', 'customer_classification_dataset.parquet', 'olist_valid_transactions.parquet', 'segment_profiles.csv', 'cancelled_invoices.parquet', 'customer_rfm.parquet', 'returns.parquet', 'valid_purchases.parquet']


In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import numpy as np
import pandas as pd

# Ուսուցման և ստուգման տվյալներ
X_train = train[feature_columns]
y_train = train["Target"]

X_val = validation[feature_columns]
y_val = validation["Target"]

# Մոդելներ
models = {
    "Dummy": DummyClassifier(strategy="prior"),

    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]

    results.append({
        "Model": name,
        "Average_Precision": average_precision_score(
            y_val, probabilities
        ),
        "ROC_AUC": roc_auc_score(y_val, probabilities)
    })

results = pd.DataFrame(results)
print(results.sort_values("Average_Precision", ascending=False))

StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 11, Finished, Available, Finished, False)

🏃 View run tender_pummelo_1z5s275w at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/eb911c88-51c6-49b2-aae3-4339f29739bf
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run frank_chin_1g1pb3m1 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/fe610b31-904e-4002-9dfc-3d9aac66ff71
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


🏃 View run lemon_stomach_gh1b5j46 at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c/runs/546bd59f-3d64-413a-82f7-66df22eec9a9
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/4fe6667e-7d48-4aab-9a98-f665ecffdc03/mlflow/#/experiments/f474d5d5-86bb-47ae-b609-44e21d9bfa0c


                 Model  Average_Precision   ROC_AUC
1  Logistic Regression           0.023255  0.608910
2        Random Forest           0.004891  0.568803
0                Dummy           0.003509  0.500000


In [9]:
# Select best model
best_model = models["Logistic Regression"]

# Predict on untouched test data
X_test = test[feature_columns]
y_test = test["Target"]

test_probabilities = best_model.predict_proba(X_test)[:, 1]

# Use threshold selected earlier on validation data
test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("FINAL TEST RESULTS")
print("Average Precision:",
      average_precision_score(y_test, test_probabilities))
print("ROC-AUC:",
      roc_auc_score(y_test, test_probabilities))
print("Precision:",
      precision_score(y_test, test_predictions, zero_division=0))
print("Recall:",
      recall_score(y_test, test_predictions, zero_division=0))
print("F1-score:",
      f1_score(y_test, test_predictions, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

# Save model
import joblib

joblib.dump(
    best_model,
    silver_path + "/final_best_model.pkl"
)

print("\nFINAL MODEL SAVED!")

StatementMeta(, e3574560-3540-4d28-b66c-d8a5393a9610, 13, Finished, Available, Finished, False)

FINAL TEST RESULTS
Average Precision: 0.017249959969568376
ROC-AUC: 0.6474327480260926
Precision: 0.07547169811320754
Recall: 0.01834862385321101
F1-score: 0.02952029520295203

Confusion Matrix:
[[58641    49]
 [  214     4]]

FINAL MODEL SAVED!
